In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


---
## Premier modèle XGBoost

In [ ]:
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

X = df_x_clean.drop(columns=['id'])
y = df_y_clean['target']

# Split Train/Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# stratify --> pour garder les proportions de classes

weights = compute_sample_weight('balanced', y_train)

# Modèle XGBoost
model_xgb = XGBClassifier(
    num_class=4,
    random_state=42
)

# Entraînement du modèle
model_xgb.fit(X_train, y_train, sample_weight=weights)

# Évaluation
y_pred = model_xgb.predict(X_test)
score_f1 = f1_score(y_test, y_pred, average='macro')

print(f"Score F1-Macro:{score_f1}")


Score F1-Macro:0.6477616379422344


In [ ]:
#entraînement de ce modèle sur tout le jeu de données d'entraînement

df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

X_train = df_x_clean.drop(columns=['id'])
y_train = df_y_clean['target']
X_test = df_x_test_clean.drop(columns=['id'])

weights = compute_sample_weight('balanced', y_train)
model_final1 = XGBClassifier(
    num_class=4,
    random_state=42
)

model_final1.fit(X_train, y_train, sample_weight=weights)

# prédiction sur le jeu de test
predictions = model_final1.predict(X_test)

# repasser des nombres aux noms de postes
postes = {
    0: "Assistant",
    1: "Executive",
    2: "Manager",
    3: "Director"
}
predictions_labels = [postes[p] for p in predictions]

# créer le fichier
predictions_final_1 = pd.DataFrame({
    'id': df_x_test_clean['id'],
    'position': predictions_labels
})
predictions_final_1.to_csv('../results/predictions_XGBoost_1.csv', index=True)

**Le score de ces prédictions est 0,26744406906832957**

on voit une forte difference entre le score obtenu en faisant le train test-split (~0,61) et sur le resultat ou on l'entraine sur toutes les données de tests (~0,26)

les explications possibles:
- surapprentissage
- le modèle peut aussi être trop sensible au découpage interne --> faire une validation croisée


--> le problème: dans le train on a plusieurs fois la meme entreprise --> le meme embeddings pour l'entreprise donc quand on fait le train/test split, on test sur des sonnées ou on a les memes entreprise sur lesquelles on s'est entrainé.
Or quand on fait le test pour soumettre des resultats, ce sont des nouvelles entreprises jamais vu donc on a un f1 score beaucoup mopins bien.

solution: donner un id aux différentes entreprises pour que les données de meme entreprises ne se retrouvent pas dans le train et dans le test et fausse le score F1.

---
## Validation croisée + création d'id pour chaque entreprise

In [5]:
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

# les 32 colonnes d'entreprise
company_cols = [f'company_feature_{i}' for i in range(32)]

#regroupe sous un même identifiant les lignes qui ont la meme entreprise
df_x_clean['company_group'] = df_x_clean[company_cols].apply(lambda row: hash(tuple(row)), axis=1)
groups_company = df_x_clean['company_group']


#on va faire une validation croisée en utilisant StratifiedGroupKFold
#le découpage respecte à la fois les groupes d'entités et l'équilibre des classes dans chaque bloc

X = df_x_clean.drop(columns=['id', 'company_group'])
y = df_y_clean['target']

cv = StratifiedGroupKFold(n_splits=5,random_state=42, shuffle=True)
scores_f1 = []

#entraînement et d'évaluation pli par pli
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups=groups_company)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    
    weights = compute_sample_weight('balanced', y_tr)
    
    model = XGBClassifier(
        num_class=4,
        random_state=42,
    )
    
    model.fit(X_tr, y_tr, sample_weight=weights)
    
    preds = model.predict(X_te)
    score = f1_score(y_te, preds, average='macro')
    scores_f1.append(score)
    
    print(f"Pli {fold + 1} - F1-Macro : {score}")

print(f"\n F1-Macro Moyen : {np.mean(scores_f1)}")


Pli 1 - F1-Macro : 0.3083925326770606
Pli 2 - F1-Macro : 0.3159698713802746
Pli 3 - F1-Macro : 0.28138466468243956
Pli 4 - F1-Macro : 0.34630460825558573
Pli 5 - F1-Macro : 0.2863803977610293

 F1-Macro Moyen : 0.30768641495127796


---
## Recherche hyperparamètres XGBoost

In [ ]:
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

# les 32 colonnes d'entreprise
company_cols = [f'company_feature_{i}' for i in range(32)]

#regroupe sous un même identifiant les lignes qui ont la meme entreprise
df_x_clean['company_group'] = df_x_clean[company_cols].apply(lambda row: hash(tuple(row)), axis=1)
groups_company = df_x_clean['company_group']

#on va faire une validation croisée en utilisant StratifiedGroupKFold
#le découpage respecte à la fois les groupes d'entités et l'équilibre des classes dans chaque bloc.

X = df_x_clean.drop(columns=['id', 'company_group'])
y = df_y_clean['target']


# 2. hyperparamètres à tester
param_test = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 4, 6, 8],
    'learning_rate': [0.03, 0.1, 0.2],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0],
    'min_child_weight': [1, 3, 5],
    'reg_lambda': [1, 5, 10]
}


sample_weights = compute_sample_weight('balanced', y)

model1 = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    random_state=42
)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# recherche aléatoire hyperparamètres
recherche = RandomizedSearchCV(
    estimator=model1,
    param_distributions=param_test,
    n_iter=20,
    scoring='f1_macro',
    cv=cv,
    random_state=42,
)
recherche.fit(X, y, groups=groups_company, sample_weight=sample_weights)


print(f"Meilleur F1-Macro Réel: {recherche.best_score_}")

print("\nMeilleurs hyperparamètres trouvés :")
print(recherche.best_params_)

--- Meilleur F1-Macro Réel (GroupKFold) : 0.3576 ---

Meilleurs hyperparamètres trouvés :
{'subsample': 0.85, 'reg_lambda': 5, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 0.7}


On va maintenant entrainer le même modèle avec les meilleurs hyperparamètres trouvés sur l'ensemble des données train 

Meilleurs hyperparamètres:  
'subsample': 0.85  
'reg_lambda': 5  
'n_estimators': 300  
'min_child_weight': 1  
'max_depth': 4  
'learning_rate': 0.03  
'colsample_bytree': 0.7  


In [ ]:
#entraînement du modéle avec les meilleurs hyperparamètres sur tout le jeu de données d'entraînement
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

X_train = df_x_clean.drop(columns=['id'])
y_train = df_y_clean['target']
X_test = df_x_test_clean.drop(columns=['id'])

weights = compute_sample_weight('balanced', y_train)
model_final2 = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    random_state=42,
    subsample=recherche.best_params_['subsample'],
    reg_lambda=recherche.best_params_['reg_lambda'],
    n_estimators=recherche.best_params_['n_estimators'],
    min_child_weight=recherche.best_params_['min_child_weight'],
    max_depth=recherche.best_params_['max_depth'],
    learning_rate=recherche.best_params_['learning_rate'],
    colsample_bytree=recherche.best_params_['colsample_bytree']
)

model_final2.fit(X_train, y_train, sample_weight=weights)

# prédiction sur le jeu de test
predictions = model_final2.predict(X_test)

# repasser des nombres aux noms de postes
postes = {
    0: "Assistant",
    1: "Executive",
    2: "Manager",
    3: "Director"
}
predictions_labels = [postes[p] for p in predictions]

# créer le fichier
predictions_final_2 = pd.DataFrame({
    'id': df_x_test_clean['id'],
    'position': predictions_labels
})
predictions_final_2.to_csv('../results/predictions_XGBoost_2.csv', index=True)


**Le score de ces predictions est 0,25182887572573764**